# Retail Demand Forecasting Pipeline
**Author:** Bawelile Gule · Data Scientist & AI Strategist  
**Stack:** Python · Prophet · XGBoost · MLflow · Plotly  
[![Portfolio](https://img.shields.io/badge/Portfolio-Bawelile.github.io-1A5276?style=flat-square)](https://Bawelile.github.io) [![LinkedIn](https://img.shields.io/badge/LinkedIn-BawelileGule1010-0A66C2?style=flat-square)](https://linkedin.com/in/BawelileGule1010)

---

## Business Problem

Retail operations live and die by forecast accuracy. Over-forecasting locks capital in unsold inventory. Under-forecasting loses revenue that can never be recovered.

This pipeline predicts daily product demand across multiple store-product combinations and translates forecast error directly into **business dollar impact** — the metric that drives operational decisions, not just model accuracy scores.

## Pipeline Architecture
```
Raw Data  →  Ingest & Validate  →  Feature Engineering  →  Train (Prophet + XGBoost)
                                                                ↓
Forward Forecast  ←  Business Impact Translation  ←  MLflow Experiment Tracking
```

**Run all cells top to bottom. Full pipeline completes in ~3 minutes on Colab.**

---
## 0 · Install dependencies

In [ ]:
%%capture
!pip install prophet mlflow xgboost plotly --quiet

---
## 1 · Imports & configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import mlflow
import xgboost as xgb
from prophet import Prophet
from sklearn.metrics import mean_squared_error
from IPython.display import display

np.random.seed(42)

# Pipeline configuration — change store/product to run on any combination
CONFIG = {
    'store'     : 'ATL-01',
    'product'   : 'Cinnamon Classic',
    'test_days' : 60,
    'horizon'   : 30,
    'avg_price' : 5.75,
}

STORES   = ['ATL-01', 'ATL-02', 'ATL-03', 'CHI-01', 'NYC-01']
PRODUCTS = ['Cinnamon Classic', 'Vegan Delight', 'Choco Fudge', 'Berry Bliss', 'Nutella Dream']
PROMO_WEEKS = [4, 13, 22, 27, 35, 44, 50]

print(f'Configuration: {CONFIG["store"]} | {CONFIG["product"]}')
print(f'Test window: {CONFIG["test_days"]} days | Forecast horizon: {CONFIG["horizon"]} days')

---
## 2 · Synthetic retail dataset

Generates 2 years of daily sales across 5 stores and 5 products.

Demand is modelled as a sum of components: `trend + yearly_seasonality + weekly_seasonality + promo_lift + holiday_lift + noise` — mirroring the additive decomposition used in time-series analysis. ~2% missing values are introduced to simulate real-world data pipeline conditions.

In [ ]:
HOLIDAYS = pd.to_datetime([
    '2022-07-04','2022-11-24','2022-12-25',
    '2023-07-04','2023-11-23','2023-12-25',
])

PRODUCT_CFG = {
    'Cinnamon Classic': {'base': 120, 'amp': 0.25, 'price': 5.50},
    'Vegan Delight'   : {'base':  85, 'amp': 0.15, 'price': 6.25},
    'Choco Fudge'     : {'base': 100, 'amp': 0.30, 'price': 5.75},
    'Berry Bliss'     : {'base':  70, 'amp': 0.40, 'price': 5.95},
    'Nutella Dream'   : {'base':  95, 'amp': 0.20, 'price': 6.50},
}

def generate_series(start, end, product, store):
    cfg   = PRODUCT_CFG[product]
    dates = pd.date_range(start, end, freq='D')
    n, t  = len(dates), np.arange(len(dates))

    trend        = cfg['base'] + 0.03 * t
    yearly       = cfg['amp'] * cfg['base'] * np.sin(2 * np.pi * t / 365)
    weekly       = 0.12 * cfg['base'] * np.sin(2 * np.pi * t / 7)
    noise        = np.random.normal(0, cfg['base'] * 0.08, n)

    df = pd.DataFrame({'date': dates})
    df['store']      = store
    df['product']    = product
    df['week']       = df['date'].dt.isocalendar().week.astype(int)
    df['is_promo']   = df['week'].isin(PROMO_WEEKS).astype(int)
    df['is_holiday'] = df['date'].isin(HOLIDAYS).astype(int)
    df['is_weekend'] = (df['date'].dt.dayofweek >= 5).astype(int)

    raw = (trend + yearly + weekly + noise
           + df['is_promo']   * cfg['base'] * np.random.uniform(0.18, 0.30)
           + df['is_holiday'] * cfg['base'] * np.random.uniform(0.40, 0.60)
           + df['is_weekend'] * cfg['base'] * 0.15)

    df['units_sold']        = np.clip(raw, 0, None).round().astype(int)
    df['inventory_on_hand'] = (df['units_sold'] * np.random.uniform(1.1, 2.5, n)).round().astype(int)
    df['revenue']           = (df['units_sold'] * cfg['price']).round(2)

    # Introduce realistic missing values (~2%)
    for col in ['units_sold', 'inventory_on_hand', 'revenue']:
        df.loc[np.random.random(n) < 0.02, col] = np.nan

    return df

frames = [generate_series('2022-01-01', '2023-12-31', p, s) for s in STORES for p in PRODUCTS]
RAW_DF = pd.concat(frames, ignore_index=True).sort_values(['store','product','date']).reset_index(drop=True)

print(f'Dataset: {len(RAW_DF):,} rows | {RAW_DF["store"].nunique()} stores | {RAW_DF["product"].nunique()} products')
print(f'Date range: {RAW_DF["date"].min().date()} to {RAW_DF["date"].max().date()}')
print(f'Missing values: {RAW_DF.isnull().sum().sum()} (~2% intentional)')

---
## 3 · Ingest & validate

Schema validation, per-group forward-fill imputation, and sanity checks. Missing values are filled within each store-product group to preserve local time-series patterns rather than replacing with global statistics.

In [ ]:
def ingest(raw):
    required = ['date','store','product','units_sold','revenue','inventory_on_hand',
                'is_promo','is_holiday','is_weekend']
    missing_cols = [c for c in required if c not in raw.columns]
    assert not missing_cols, f'Missing columns: {missing_cols}'

    df = raw.copy()
    df['store']   = df['store'].astype('category')
    df['product'] = df['product'].astype('category')

    null_report = df.isnull().sum()
    print('Missing values before imputation:')
    print(null_report[null_report > 0].to_string())

    # Forward-fill then backward-fill within each store-product group
    # Preserves local demand patterns rather than using global statistics
    numeric = ['units_sold', 'inventory_on_hand', 'revenue']
    df[numeric] = (
        df.groupby(['store','product'], observed=True)[numeric]
        .transform(lambda s: s.ffill().bfill())
    )
    for col in numeric:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())

    assert (df['units_sold'] >= 0).all(), 'Negative units_sold detected'
    assert (df['revenue']    >= 0).all(), 'Negative revenue detected'
    print('\nIngestion complete — schema valid, nulls resolved, sanity checks passed.')
    return df


def get_series(df, store, product):
    """Extract a single store-product series with no date gaps."""
    s = (df[(df['store'] == store) & (df['product'] == product)]
         .copy().set_index('date').sort_index())
    full_idx = pd.date_range(s.index.min(), s.index.max(), freq='D')
    s = s.reindex(full_idx)
    s['units_sold'] = s['units_sold'].fillna(0)
    s.index.name = 'date'
    return s.reset_index()


CLEAN_DF = ingest(RAW_DF)
SERIES   = get_series(CLEAN_DF, CONFIG['store'], CONFIG['product'])
print(f'\nSeries: {CONFIG["store"]} | {CONFIG["product"]} — {len(SERIES)} rows')

---
## 4 · Exploratory data analysis

In [ ]:
# Sales distribution across stores and products
store_avg   = CLEAN_DF.groupby('store',   observed=True)['units_sold'].mean().sort_values()
product_avg = CLEAN_DF.groupby('product', observed=True)['units_sold'].mean().sort_values()

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    'Avg daily units — by store', 'Avg daily units — by product'
))
fig.add_trace(go.Bar(x=store_avg.values,   y=store_avg.index,   orientation='h', marker_color='#1A5276'), row=1, col=1)
fig.add_trace(go.Bar(x=product_avg.values, y=product_avg.index, orientation='h', marker_color='#2E86C1'), row=1, col=2)
fig.update_layout(height=300, showlegend=False, title_text='Sales distribution overview')
fig.show()

In [ ]:
# Historical series with promotional and holiday annotations
s = SERIES.copy()
fig = go.Figure()
fig.add_trace(go.Scatter(x=s['date'], y=s['units_sold'], mode='lines',
                         name='Units sold', line=dict(color='#1A5276', width=1.2)))
fig.add_trace(go.Scatter(x=s[s['is_promo']==1]['date'], y=s[s['is_promo']==1]['units_sold'],
                         mode='markers', name='Promo day',
                         marker=dict(color='#F39C12', size=5, symbol='triangle-up')))
fig.add_trace(go.Scatter(x=s[s['is_holiday']==1]['date'], y=s[s['is_holiday']==1]['units_sold'],
                         mode='markers', name='Holiday',
                         marker=dict(color='#E74C3C', size=9, symbol='star')))
fig.update_layout(title=f'Historical series — {CONFIG["store"]} | {CONFIG["product"]}',
                  height=350, plot_bgcolor='white', legend=dict(orientation='h', y=1.12))
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()

In [ ]:
# Weekly seasonality and promotional lift analysis
s['day_name'] = s['date'].dt.day_name()
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_avg   = s.groupby('day_name')['units_sold'].mean().reindex(dow_order)

promo_df = (
    CLEAN_DF.groupby(['product','is_promo'], observed=True)['units_sold']
    .mean().unstack().rename(columns={0:'Non-promo', 1:'Promo'})
)
promo_df['Lift (%)'] = ((promo_df['Promo'] - promo_df['Non-promo']) / promo_df['Non-promo'] * 100).round(1)

fig = make_subplots(rows=1, cols=2, subplot_titles=('Weekly demand pattern', 'Promotional lift by product'))
fig.add_trace(go.Bar(x=dow_order, y=dow_avg.values, marker_color='#2E86C1'), row=1, col=1)
fig.add_trace(go.Bar(y=promo_df.index, x=promo_df['Lift (%)'].sort_values(),
                     orientation='h', marker_color='#E67E22'), row=1, col=2)
fig.update_layout(height=320, showlegend=False)
fig.show()

weekday_avg = dow_avg[['Monday','Tuesday','Wednesday','Thursday','Friday']].mean()
weekend_avg = dow_avg[['Saturday','Sunday']].mean()
print(f'Weekend uplift: {(weekend_avg/weekday_avg - 1)*100:.1f}%')
print(f'Avg promotional lift: {promo_df["Lift (%)"].mean():.1f}%')

---
## 5 · Feature engineering

19 features across 5 groups: calendar signals, lag features (t-7, t-14, t-28), rolling statistics (7-day and 28-day mean/std/max), linear trend index, and external event flags.

All rolling features use `.shift(1)` before the rolling window to prevent data leakage — the model never sees today's value when computing today's features.

In [ ]:
def build_features(df):
    df = df.copy().sort_values('date').reset_index(drop=True)

    # Calendar features
    df['day_of_week']    = df['date'].dt.dayofweek
    df['day_of_month']   = df['date'].dt.day
    df['week_of_year']   = df['date'].dt.isocalendar().week.astype(int)
    df['month']          = df['date'].dt.month
    df['quarter']        = df['date'].dt.quarter
    df['is_month_end']   = df['date'].dt.is_month_end.astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['t']              = np.arange(len(df))

    # Lag features — same-day-last-week is the strongest retail predictor
    for lag in [7, 14, 28]:
        df[f'lag_{lag}'] = df['units_sold'].shift(lag)

    # Rolling statistics — .shift(1) prevents leakage of current observation
    for window in [7, 28]:
        df[f'rolling_mean_{window}'] = df['units_sold'].shift(1).rolling(window, min_periods=3).mean()
        df[f'rolling_std_{window}']  = df['units_sold'].shift(1).rolling(window, min_periods=3).std()
    df['rolling_max_7'] = df['units_sold'].shift(1).rolling(7, min_periods=3).max()

    # Price proxy derived from revenue / units_sold
    df['avg_price'] = np.where(df['units_sold'] > 0, df['revenue'] / df['units_sold'], np.nan)
    df['avg_price'] = df['avg_price'].ffill().bfill()

    return df.dropna(subset=['lag_7','lag_14','lag_28']).reset_index(drop=True)


FEATURE_COLS = [
    'day_of_week','day_of_month','week_of_year','month','quarter',
    'is_month_end','is_month_start','t',
    'lag_7','lag_14','lag_28',
    'rolling_mean_7','rolling_std_7','rolling_mean_28','rolling_std_28','rolling_max_7',
    'avg_price','is_promo','is_holiday','is_weekend',
]

FEAT_DF = build_features(SERIES)
print(f'Feature matrix: {FEAT_DF.shape[0]} rows × {len(FEATURE_COLS)} features')

# Feature correlation with target
corr = FEAT_DF[FEATURE_COLS + ['units_sold']].corr()['units_sold'].drop('units_sold').sort_values()
fig  = go.Figure(go.Bar(x=corr.values, y=corr.index, orientation='h',
                        marker_color=['#E74C3C' if v < 0 else '#1A5276' for v in corr.values]))
fig.add_vline(x=0, line_dash='dash', line_color='gray', line_width=1)
fig.update_layout(title='Feature correlation with units_sold', height=500, plot_bgcolor='white')
fig.show()
print(f'Strongest predictor: {corr.abs().idxmax()} (r = {corr.abs().max():.3f})')

---
## 6 · Temporal train / test split

Chronological split — the last 60 days form the held-out test set. Time-series data must never be randomly shuffled: doing so would leak future observations into training, producing falsely optimistic evaluation metrics.

In [ ]:
def temporal_split(df, test_days=60):
    cutoff = df['date'].max() - pd.Timedelta(days=test_days)
    return df[df['date'] <= cutoff].copy(), df[df['date'] > cutoff].copy()

TRAIN, TEST = temporal_split(FEAT_DF, test_days=CONFIG['test_days'])
print(f'Train: {len(TRAIN):,} rows  ({TRAIN["date"].min().date()} → {TRAIN["date"].max().date()})')
print(f'Test : {len(TEST):,} rows  ({TEST["date"].min().date()} → {TEST["date"].max().date()})')
assert TRAIN['date'].max() < TEST['date'].min(), 'Temporal leakage detected'
print('No temporal leakage confirmed.')

---
## 7 · Model training

**Prophet** captures trend and seasonality natively using Fourier decomposition. Custom regressors (is_promo, is_holiday, is_weekend) add known external demand drivers.

**XGBoost** exploits the engineered lag and rolling features to capture local demand patterns that Prophet cannot access directly.

**Ensemble** averages both models — their errors are largely uncorrelated, so the combined prediction consistently outperforms either model alone. All runs are tracked in MLflow.

In [ ]:
mlflow.set_experiment('retail-demand-forecasting')

def to_prophet_df(df):
    return (df[['date','units_sold','is_promo','is_holiday','is_weekend']]
            .rename(columns={'date':'ds','units_sold':'y'})
            .sort_values('ds').reset_index(drop=True))

# Prophet
prophet_params = {
    'changepoint_prior_scale': 0.05,
    'seasonality_prior_scale': 10.0,
    'seasonality_mode'       : 'multiplicative',
    'yearly_seasonality'     : True,
    'weekly_seasonality'     : True,
}

with mlflow.start_run(run_name=f'prophet_{CONFIG["store"]}_{CONFIG["product"].replace(" ","_")}'):
    mlflow.log_params({**prophet_params, 'model':'prophet', 'store':CONFIG['store'], 'product':CONFIG['product']})
    prophet_model = Prophet(**prophet_params)
    for reg in ['is_promo','is_holiday','is_weekend']:
        prophet_model.add_regressor(reg)
    prophet_model.fit(to_prophet_df(TRAIN))
    PROPHET_PREDS = prophet_model.predict(
        to_prophet_df(TEST)[['ds','is_promo','is_holiday','is_weekend']]
    )['yhat'].clip(0).values
    ACTUALS = to_prophet_df(TEST)['y'].values
    p_rmse = float(np.sqrt(mean_squared_error(ACTUALS, PROPHET_PREDS)))
    p_mape = float(np.mean(np.abs((ACTUALS - PROPHET_PREDS) / (ACTUALS + 1e-8))))
    mlflow.log_metrics({'rmse': round(p_rmse,4), 'mape': round(p_mape,4)})

print(f'Prophet  — RMSE: {p_rmse:.2f}  MAPE: {p_mape:.2%}')

In [ ]:
# XGBoost
xgb_params = {
    'n_estimators':400, 'max_depth':5, 'learning_rate':0.05,
    'subsample':0.85, 'colsample_bytree':0.80, 'min_child_weight':3,
    'reg_alpha':0.1, 'reg_lambda':1.0, 'random_state':42,
    'objective':'reg:squarederror',
}

with mlflow.start_run(run_name=f'xgboost_{CONFIG["store"]}_{CONFIG["product"].replace(" ","_")}'):
    mlflow.log_params({**xgb_params, 'model':'xgboost', 'store':CONFIG['store'], 'product':CONFIG['product']})
    xgb_model = xgb.XGBRegressor(**xgb_params)
    xgb_model.fit(TRAIN[FEATURE_COLS], TRAIN['units_sold'],
                  eval_set=[(TEST[FEATURE_COLS], TEST['units_sold'])], verbose=False)
    XGB_PREDS = xgb_model.predict(TEST[FEATURE_COLS]).clip(0)
    x_rmse = float(np.sqrt(mean_squared_error(ACTUALS, XGB_PREDS)))
    x_mape = float(np.mean(np.abs((ACTUALS - XGB_PREDS) / (ACTUALS + 1e-8))))
    mlflow.log_metrics({'rmse': round(x_rmse,4), 'mape': round(x_mape,4)})
    FEATURE_IMPORTANCE = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

print(f'XGBoost  — RMSE: {x_rmse:.2f}  MAPE: {x_mape:.2%}')

In [ ]:
# Ensemble — equal-weighted average
ENS_PREDS = (0.5 * PROPHET_PREDS + 0.5 * XGB_PREDS).clip(0)
e_rmse    = float(np.sqrt(mean_squared_error(ACTUALS, ENS_PREDS)))
e_mape    = float(np.mean(np.abs((ACTUALS - ENS_PREDS) / (ACTUALS + 1e-8))))

with mlflow.start_run(run_name=f'ensemble_{CONFIG["store"]}_{CONFIG["product"].replace(" ","_")}'):
    mlflow.log_params({'model':'ensemble', 'store':CONFIG['store'], 'product':CONFIG['product']})
    mlflow.log_metrics({'rmse': round(e_rmse,4), 'mape': round(e_mape,4)})

comparison = pd.DataFrame({
    'Model'  : ['Prophet', 'XGBoost', 'Ensemble'],
    'RMSE'   : [round(p_rmse,2), round(x_rmse,2), round(e_rmse,2)],
    'MAPE'   : [f'{p_mape:.2%}', f'{x_mape:.2%}', f'{e_mape:.2%}'],
}).set_index('Model')

print('=== Model comparison ===')
display(comparison)

---
## 8 · Evaluation & business impact

Forecast error is translated into operational dollar impact — the metric that matters to operations and finance teams, not just data scientists.

In [ ]:
test_dates = TEST['date'].values

fig = go.Figure()
fig.add_trace(go.Scatter(x=test_dates, y=ACTUALS,       mode='lines', name='Actual',    line=dict(color='#1A5276', width=2)))
fig.add_trace(go.Scatter(x=test_dates, y=PROPHET_PREDS, mode='lines', name='Prophet',   line=dict(color='#8E44AD', width=1.5, dash='dot')))
fig.add_trace(go.Scatter(x=test_dates, y=XGB_PREDS,     mode='lines', name='XGBoost',   line=dict(color='#27AE60', width=1.5, dash='dash')))
fig.add_trace(go.Scatter(x=test_dates, y=ENS_PREDS,     mode='lines', name='Ensemble',  line=dict(color='#E67E22', width=2)))
fig.update_layout(title='Forecast vs Actuals — test period', height=380,
                  plot_bgcolor='white', legend=dict(orientation='h', y=1.12))
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()

In [ ]:
# Feature importance
fi_top = FEATURE_IMPORTANCE.head(12)
fig = go.Figure(go.Bar(x=fi_top.values[::-1], y=fi_top.index[::-1], orientation='h', marker_color='#1A5276'))
fig.update_layout(title='XGBoost — top 12 feature importances', height=360, plot_bgcolor='white')
fig.show()
print(f'Top 3 features: {FEATURE_IMPORTANCE.head(3).index.tolist()}')

In [ ]:
# Business dollar impact translation
price    = CONFIG['avg_price']
errors   = np.abs(ACTUALS - ENS_PREDS)
over_fc  = np.sum(np.clip(ENS_PREDS - ACTUALS, 0, None))
under_fc = np.sum(np.clip(ACTUALS - ENS_PREDS,   0, None))

print('=' * 55)
print(f'  BUSINESS IMPACT — {CONFIG["store"]} | {CONFIG["product"]}')
print('=' * 55)
print(f'  Ensemble MAPE            : {e_mape:.2%}')
print(f'  Ensemble RMSE            : {e_rmse:.2f} units/day')
print(f'  Avg daily error          : {errors.mean():.1f} units  (${errors.mean()*price:.2f}/day)')
print(f'  Over-forecast waste      : {over_fc:,.0f} units  (${over_fc*price:,.2f})')
print(f'  Revenue at risk          : {under_fc:,.0f} units  (${under_fc*price:,.2f})')
print(f'  Monthly revenue impact   : ~${errors.mean()*price*30:,.2f}')
print('=' * 55)

---
## 9 · Forward forecast

Generates a 30-day forward forecast beyond the last known date. Future lag features are derived from the known historical tail. Uncertainty bounds of ±15% are applied as a conservative operational planning envelope.

In [ ]:
def build_future_frame(feat_df, horizon):
    last_date    = feat_df['date'].max()
    future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=horizon, freq='D')
    known_vals   = feat_df['units_sold'].values

    future = pd.DataFrame({'date': future_dates})
    future['day_of_week']    = future['date'].dt.dayofweek
    future['day_of_month']   = future['date'].dt.day
    future['week_of_year']   = future['date'].dt.isocalendar().week.astype(int)
    future['month']          = future['date'].dt.month
    future['quarter']        = future['date'].dt.quarter
    future['is_month_end']   = future['date'].dt.is_month_end.astype(int)
    future['is_month_start'] = future['date'].dt.is_month_start.astype(int)
    future['is_weekend']     = (future['date'].dt.dayofweek >= 5).astype(int)
    future['is_holiday']     = 0
    future['is_promo']       = future['week_of_year'].isin(PROMO_WEEKS).astype(int)
    future['t']              = np.arange(len(feat_df) + 1, len(feat_df) + 1 + horizon)

    for lag in [7, 14, 28]:
        future[f'lag_{lag}'] = [
            known_vals[min(max(len(known_vals) - lag + i, 0), len(known_vals) - 1)]
            for i in range(horizon)
        ]

    tail_7, tail_28 = known_vals[-7:], known_vals[-28:]
    future['rolling_mean_7']  = np.mean(tail_7)
    future['rolling_std_7']   = np.std(tail_7)
    future['rolling_mean_28'] = np.mean(tail_28)
    future['rolling_std_28']  = np.std(tail_28)
    future['rolling_max_7']   = np.max(tail_7)

    recent = feat_df.tail(30)
    total_units = recent['units_sold'].replace(0, np.nan).sum()
    future['avg_price'] = (recent['revenue'].sum() / total_units) if total_units > 0 else CONFIG['avg_price']

    return future


FUTURE       = build_future_frame(FEAT_DF, CONFIG['horizon'])
FUTURE_PREDS = xgb_model.predict(FUTURE[FEATURE_COLS]).clip(0)
uncertainty  = FUTURE_PREDS * 0.15
promo_mask   = FUTURE['is_promo'] == 1
tail_hist    = FEAT_DF.tail(60)

fig = go.Figure()
fig.add_trace(go.Scatter(x=tail_hist['date'], y=tail_hist['units_sold'],
                         mode='lines', name='Historical (last 60d)', line=dict(color='#1A5276', width=1.5)))
fig.add_trace(go.Scatter(
    x=pd.concat([FUTURE['date'], FUTURE['date'].iloc[::-1]]),
    y=np.concatenate([(FUTURE_PREDS + uncertainty), (FUTURE_PREDS - uncertainty)[::-1]]),
    fill='toself', fillcolor='rgba(230,126,34,0.15)',
    line=dict(color='rgba(255,255,255,0)'), name='\u00b115% uncertainty'
))
fig.add_trace(go.Scatter(x=FUTURE['date'], y=FUTURE_PREDS.round(0),
                         mode='lines+markers', name=f'{CONFIG["horizon"]}-day forecast',
                         line=dict(color='#E67E22', width=2), marker=dict(size=4)))
if promo_mask.any():
    fig.add_trace(go.Scatter(x=FUTURE[promo_mask]['date'], y=FUTURE_PREDS[promo_mask.values],
                             mode='markers', name='Promo day',
                             marker=dict(color='#F39C12', size=10, symbol='star')))
fig.update_layout(title=f'{CONFIG["horizon"]}-day forward forecast — {CONFIG["store"]} | {CONFIG["product"]}',
                  height=400, plot_bgcolor='white', legend=dict(orientation='h', y=1.12))
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()

print(f'Forecast: {CONFIG["horizon"]} days from {FUTURE["date"].min().date()}')
print(f'  Avg daily demand  : {FUTURE_PREDS.mean():.1f} units')
print(f'  Peak day          : {FUTURE["date"].iloc[FUTURE_PREDS.argmax()].date()} ({FUTURE_PREDS.max():.0f} units)')
print(f'  Promo days ahead  : {promo_mask.sum()}')
print(f'  Projected revenue : ${(FUTURE_PREDS * CONFIG["avg_price"]).sum():,.2f}')

---
## 10 · MLflow experiment summary

In [ ]:
runs = mlflow.search_runs(experiment_names=['retail-demand-forecasting'])
if not runs.empty:
    cols = [c for c in ['tags.mlflow.runName','metrics.rmse','metrics.mape'] if c in runs.columns]
    summary = (runs[cols]
               .rename(columns={'tags.mlflow.runName':'Run','metrics.rmse':'RMSE','metrics.mape':'MAPE'})
               .dropna(subset=['RMSE']).sort_values('RMSE'))
    summary['MAPE'] = summary['MAPE'].apply(lambda x: f'{x:.2%}' if pd.notna(x) else '-')
    summary['RMSE'] = summary['RMSE'].round(3)
    print('=== MLflow experiment runs (sorted by RMSE) ===')
    display(summary)

---
## 11 · Pipeline summary

In [ ]:
print('=' * 58)
print('  RETAIL DEMAND FORECASTING PIPELINE — COMPLETE')
print('=' * 58)
print(f'  Store / Product  : {CONFIG["store"]} | {CONFIG["product"]}')
print(f'  Features         : {len(FEATURE_COLS)} engineered features')
print(f'  Top feature      : {FEATURE_IMPORTANCE.index[0]}')
print()
print(f'  Prophet   RMSE: {p_rmse:>7.2f}   MAPE: {p_mape:.2%}')
print(f'  XGBoost   RMSE: {x_rmse:>7.2f}   MAPE: {x_mape:.2%}')
print(f'  Ensemble  RMSE: {e_rmse:>7.2f}   MAPE: {e_mape:.2%}  <- best')
print()
print(f'  Avg daily error  : {errors.mean():.1f} units  (${errors.mean()*price:.2f}/day)')
print(f'  Revenue at risk  : ${under_fc*price:,.2f}')
print(f'  Projected revenue: ${(FUTURE_PREDS*price).sum():,.2f} ({CONFIG["horizon"]} days)')
print('=' * 58)